# 📊 Clase 3 — Detección y Tratamiento de Outliers (Python)

## Aplicaciones de Minería de Datos a Economía y Finanzas
### Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

**Docente:** Dr. Darío Ezequiel Díaz · drdarioezequieldiaz@gmail.com

**Fecha:** Jueves 23 de abril de 2026

---

### Objetivos

1. **Detectar** outliers con cuatro métodos: IQR, z-score modificado (MAD), Mahalanobis e Isolation Forest.
2. **Clasificar** outliers: error de captura vs. genuino vs. informativo.
3. **Tratar** con winsorización, transformación logarítmica, capping y variable indicadora.
4. **Evaluar** el impacto de cada tratamiento sobre la distribución y la capacidad predictiva.
5. **Contextualizar** la detección por segmento (IQR segmentado).

> **Referencia teórica:** Apunte Clase 2, Parte II — *Registros Duplicados e Inconsistencias* (§9: Detección estadística de inconsistencias).


---\n## 1. Configuración del entorno

In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. CONFIGURACIÓN
# ══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.ensemble import IsolationForest
from sklearn.covariance import EllipticEnvelope
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from scipy import stats
from scipy.stats import median_abs_deviation
from scipy.stats.mstats import winsorize
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

COL_GOOD, COL_BAD, COL_ACC = '#2a9d8f', '#e76f51', '#f4a261'

print("✅ Entorno configurado")


---\n## 2. Carga del dataset

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. CARGA DEL DATASET
# ══════════════════════════════════════════════════════════════

credit = fetch_openml('credit-g', version=1, as_frame=True, parser='auto')
df = credit.frame.copy()

num_vars = ['duration', 'credit_amount', 'installment_commitment',
            'residence_since', 'age', 'existing_credits', 'num_dependents']

le = LabelEncoder()
y = le.fit_transform(df['class'])

print(f"✅ Dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"   Variables numéricas a analizar: {len(num_vars)}")


---
## 3. Método IQR (Tukey Fences)

El método más clásico: un valor es outlier si cae fuera de $[Q_1 - k \cdot IQR,\; Q_3 + k \cdot IQR]$ con $k = 1.5$ (estándar) o $k = 3$ (estricto).


In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. DETECCIÓN POR IQR
# ══════════════════════════════════════════════════════════════

def detect_iqr(series, k=1.5):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - k * IQR, Q3 + k * IQR
    mask = (series < lower) | (series > upper)
    return mask, lower, upper

print("=" * 70)
print("DETECCIÓN DE OUTLIERS POR IQR (k=1.5)")
print("=" * 70)
print(f"{'Variable':<30s} {'Outliers':>8s} {'%':>7s} {'Lím.inf.':>10s} {'Lím.sup.':>10s}")
print("-" * 70)

outlier_counts = {}
for v in num_vars:
    mask, lo, hi = detect_iqr(df[v].astype(float))
    n_out = mask.sum()
    pct = n_out / len(df) * 100
    outlier_counts[v] = n_out
    flag = " ⚠" if pct > 5 else ""
    print(f"  {v:<28s} {n_out:>8d} {pct:>6.1f}% {lo:>10.1f} {hi:>10.1f}{flag}")


In [ ]:
# Visualización: boxplots con outliers resaltados
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
key_vars = ['credit_amount', 'age', 'duration']

for ax, v in zip(axes, key_vars):
    mask_iqr, lo, hi = detect_iqr(df[v].astype(float))
    data = df[v].astype(float)

    ax.boxplot(data, vert=True, patch_artist=True,
               boxprops=dict(facecolor=COL_GOOD, alpha=0.4),
               medianprops=dict(color=COL_BAD, linewidth=2))
    # Resaltar outliers
    outliers = data[mask_iqr]
    ax.scatter([1]*len(outliers), outliers, c=COL_BAD, s=20, alpha=0.6, zorder=5, label=f'Outliers ({len(outliers)})')
    ax.axhline(lo, color=COL_ACC, linestyle='--', linewidth=0.8, label=f'Lím. inf. = {lo:.0f}')
    ax.axhline(hi, color=COL_ACC, linestyle='--', linewidth=0.8, label=f'Lím. sup. = {hi:.0f}')
    ax.set_title(v, fontweight='bold')
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Detección IQR (k=1.5) — Outliers resaltados en rojo', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


### 📝 Interpretación

- **`credit_amount`:** 53 outliers (5.3%) — montos elevados que exceden $Q_3 + 1.5 \cdot IQR$. Son clientes *high-value*: **outliers genuinos**, no errores. Eliminarlos sesgaría el modelo hacia créditos de bajo monto.
- **`age`:** Solo 3 outliers — personas mayores de ~64 años. Outliers genuinos e informativos: la edad avanzada puede ser factor de riesgo.
- **`num_dependents`:** 145 outliers (14.5%) — artefacto de la escala binaria (1 o 2). No son outliers reales sino una limitación del método IQR en variables discretas de baja cardinalidad.

---
## 4. Z-score modificado (MAD)


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. Z-SCORE MODIFICADO (MAD)
# ══════════════════════════════════════════════════════════════

def detect_zscore_mad(series, threshold=3.5):
    med = np.median(series)
    mad = median_abs_deviation(series)
    if mad == 0:
        return pd.Series(False, index=series.index), pd.Series(0, index=series.index)
    m_scores = 0.6745 * (series - med) / mad
    mask = np.abs(m_scores) > threshold
    return mask, m_scores

print("=" * 70)
print("DETECCIÓN POR Z-SCORE MODIFICADO (MAD) — |M| > 3.5")
print("=" * 70)
print(f"{'Variable':<30s} {'Outliers':>8s} {'%':>7s} {'M_max':>8s}")
print("-" * 55)

for v in num_vars:
    mask, scores = detect_zscore_mad(df[v].astype(float))
    n_out = mask.sum()
    pct = n_out / len(df) * 100
    m_max = np.abs(scores).max()
    print(f"  {v:<28s} {n_out:>8d} {pct:>6.1f}% {m_max:>8.2f}")


In [ ]:
# Comparación visual: IQR vs. z-score MAD en credit_amount
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

data_ca = df['credit_amount'].astype(float)

# IQR
mask_iqr, _, _ = detect_iqr(data_ca)
axes[0].hist(data_ca[~mask_iqr], bins=40, color=COL_GOOD, alpha=0.7, edgecolor='white', label='Normal')
axes[0].hist(data_ca[mask_iqr], bins=20, color=COL_BAD, alpha=0.8, edgecolor='white', label=f'Outlier IQR ({mask_iqr.sum()})')
axes[0].set_title('IQR (k=1.5)', fontweight='bold')
axes[0].legend()

# Z-score MAD
mask_mad, _ = detect_zscore_mad(data_ca)
axes[1].hist(data_ca[~mask_mad], bins=40, color=COL_GOOD, alpha=0.7, edgecolor='white', label='Normal')
axes[1].hist(data_ca[mask_mad], bins=20, color=COL_BAD, alpha=0.8, edgecolor='white', label=f'Outlier MAD ({mask_mad.sum()})')
axes[1].set_title('Z-score modificado (MAD, |M| > 3.5)', fontweight='bold')
axes[1].legend()

plt.suptitle('credit_amount — Comparación IQR vs. Z-score MAD', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n  IQR detecta {mask_iqr.sum()} outliers vs. MAD detecta {mask_mad.sum()} outliers")
print(f"  Coinciden: {(mask_iqr & mask_mad).sum()} | Solo IQR: {(mask_iqr & ~mask_mad).sum()} | Solo MAD: {(~mask_iqr & mask_mad).sum()}")


### 📝 Interpretación

El z-score modificado basado en MAD es **más robusto** que el IQR frente a la propia presencia de outliers (breakdown point = 50% vs. 25%). En variables con asimetría pronunciada como `credit_amount`, la MAD detecta menos falsos positivos porque la mediana y la MAD son menos sensibles a los valores extremos que $Q_1$, $Q_3$ y el IQR.

---
## 5. Detección multivariada


In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. DETECCIÓN MULTIVARIADA
# ══════════════════════════════════════════════════════════════

X_num = df[num_vars].astype(float)

# 5.1 Distancia de Mahalanobis (via EllipticEnvelope)
ee = EllipticEnvelope(contamination=0.05, random_state=42)
mahal_pred = ee.fit_predict(X_num)
mask_mahal = mahal_pred == -1

# 5.2 Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42, n_estimators=200)
iso_pred = iso.fit_predict(X_num)
mask_iso = iso_pred == -1

print("=" * 60)
print("DETECCIÓN MULTIVARIADA (contamination = 5%)")
print("=" * 60)
print(f"  Mahalanobis (EllipticEnvelope): {mask_mahal.sum()} outliers ({mask_mahal.mean()*100:.1f}%)")
print(f"  Isolation Forest:               {mask_iso.sum()} outliers ({mask_iso.mean()*100:.1f}%)")
print(f"  Coinciden:                       {(mask_mahal & mask_iso).sum()}")

# Scatter: credit_amount vs. age con outliers resaltados
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
for ax, mask, title in zip(axes, [mask_mahal, mask_iso], ['Mahalanobis', 'Isolation Forest']):
    ax.scatter(df['age'].astype(float)[~mask], df['credit_amount'].astype(float)[~mask],
               c=COL_GOOD, s=15, alpha=0.4, label='Normal')
    ax.scatter(df['age'].astype(float)[mask], df['credit_amount'].astype(float)[mask],
               c=COL_BAD, s=30, alpha=0.7, edgecolors='black', linewidth=0.5, label='Outlier')
    ax.set_xlabel('Edad'); ax.set_ylabel('Monto del crédito')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Detección multivariada: credit_amount vs. age', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


### 📝 Interpretación

Los métodos multivariados detectan outliers que los métodos univariados pierden: un valor que es normal en cada variable por separado puede ser anómalo en la combinación. Por ejemplo, un crédito de alto monto para un solicitante muy joven puede no ser outlier en `credit_amount` ni en `age` individualmente, pero sí en el espacio bidimensional.

- **Mahalanobis** (vía EllipticEnvelope): asume normalidad multivariada. Funciona bien cuando la distribución conjunta es aproximadamente elíptica.
- **Isolation Forest**: no paramétrico, basado en árboles. Aísla outliers con pocas particiones. Robusto frente a distribuciones no normales.

---
## 6. IQR contextualizado por segmento


In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. IQR CONTEXTUALIZADO POR SEGMENTO
# ══════════════════════════════════════════════════════════════

# Crear tramos de edad
df['tramo_edad'] = pd.cut(df['age'].astype(float),
                           bins=[18, 25, 35, 45, 55, 100],
                           labels=['18-25', '26-35', '36-45', '46-55', '56+'])

print("=" * 70)
print("IQR CONTEXTUALIZADO: credit_amount por tramo de edad")
print("=" * 70)
print(f"{'Tramo':<10s} {'N':>6s} {'Outliers':>8s} {'%':>7s} {'Lím.sup.':>10s}")
print("-" * 45)

total_global = 0
total_context = 0

for tramo in df['tramo_edad'].cat.categories:
    sub = df[df['tramo_edad'] == tramo]['credit_amount'].astype(float)
    mask_ctx, _, hi = detect_iqr(sub)
    mask_global, _, _ = detect_iqr(df['credit_amount'].astype(float))
    n_ctx = mask_ctx.sum()
    total_context += n_ctx
    print(f"  {tramo:<8s} {len(sub):>6d} {n_ctx:>8d} {n_ctx/len(sub)*100:>6.1f}% {hi:>10.0f}")

mask_global_all, _, _ = detect_iqr(df['credit_amount'].astype(float))
print(f"\n  IQR global detecta {mask_global_all.sum()} outliers")
print(f"  IQR contextualizado detecta {total_context} outliers en total")
print(f"  Diferencia: {abs(mask_global_all.sum() - total_context)} registros")


### 📝 Interpretación

La contextualización por segmento cambia sustancialmente la detección: el límite superior de `credit_amount` para jóvenes (18-25) es significativamente menor que para adultos (46-55), porque los jóvenes solicitan montos menores. Un crédito de $10.000 es normal para un adulto pero puede ser outlier para un joven de 20 años. El IQR global pierde esta distinción.

---
## 7. Estrategias de tratamiento


In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. ESTRATEGIAS DE TRATAMIENTO
# ══════════════════════════════════════════════════════════════

data_ca = df['credit_amount'].astype(float).copy()

# T1: Winsorización al percentil 1/99
winsorized = pd.Series(winsorize(data_ca, limits=[0.01, 0.01]), index=data_ca.index)

# T2: Transformación logarítmica
log_transformed = np.log1p(data_ca)

# T3: Capping al percentil 5/95
p5, p95 = data_ca.quantile(0.05), data_ca.quantile(0.95)
capped = data_ca.clip(lower=p5, upper=p95)

# T4: Variable indicadora (flag)
mask_out, _, _ = detect_iqr(data_ca)
df['is_outlier_ca'] = mask_out.astype(int)

# Comparación visual
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

axes[0, 0].hist(data_ca, bins=50, color=COL_GOOD, alpha=0.7, edgecolor='white')
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axvline(data_ca.mean(), color=COL_BAD, linestyle='--', label=f'Media = {data_ca.mean():.0f}')
axes[0, 0].legend()

axes[0, 1].hist(winsorized, bins=50, color=COL_ACC, alpha=0.7, edgecolor='white')
axes[0, 1].set_title(f'Winsorizado (P1/P99)', fontweight='bold')
axes[0, 1].axvline(winsorized.mean(), color=COL_BAD, linestyle='--', label=f'Media = {winsorized.mean():.0f}')
axes[0, 1].legend()

axes[1, 0].hist(log_transformed, bins=50, color='#8B5CF6', alpha=0.7, edgecolor='white')
axes[1, 0].set_title('log(1 + credit_amount)', fontweight='bold')
axes[1, 0].axvline(log_transformed.mean(), color=COL_BAD, linestyle='--')

axes[1, 1].hist(capped, bins=50, color='#3B82F6', alpha=0.7, edgecolor='white')
axes[1, 1].set_title(f'Capping (P5={p5:.0f} / P95={p95:.0f})', fontweight='bold')
axes[1, 1].axvline(capped.mean(), color=COL_BAD, linestyle='--', label=f'Media = {capped.mean():.0f}')
axes[1, 1].legend()

plt.suptitle('Estrategias de tratamiento de outliers en credit_amount', fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("\nEstadísticos comparativos:")
print(f"  {'Método':<25s} {'Media':>10s} {'Mediana':>10s} {'Desvío':>10s} {'Asimetría':>10s}")
print("-" * 68)
for name, series in [("Original", data_ca), ("Winsorizado P1/P99", winsorized),
                     ("Log(1+x)", log_transformed), ("Capping P5/P95", capped)]:
    print(f"  {name:<25s} {series.mean():>10.1f} {series.median():>10.1f} {series.std():>10.1f} {series.skew():>10.3f}")


### 📝 Interpretación

Cada estrategia tiene un perfil distinto:

- **Winsorización (P1/P99):** Recorta los extremos al percentil elegido. Preserva todos los registros, reduce ligeramente la media y el desvío. Mínima distorsión.
- **Transformación log:** Reduce drásticamente la asimetría (skewness baja de ~1.9 a ~0.1). Ideal para variables monetarias. El modelo trabaja con la escala transformada; la interpretación requiere exponenciar.
- **Capping (P5/P95):** Más agresivo que la winsorización. Fija límites duros de negocio. Usado en scoring regulatorio donde se requieren rangos predefinidos.
- **Variable indicadora:** No modifica el dato; el modelo decide si el flag `is_outlier` aporta información predictiva.

---
## 8. Impacto en scoring


In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. IMPACTO EN SCORING: AUC POR ESTRATEGIA DE TRATAMIENTO
# ══════════════════════════════════════════════════════════════

print("=" * 60)
print("IMPACTO EN SCORING: AUC-ROC (5-fold CV)")
print("=" * 60)

X_base = df[num_vars].astype(float)

strategies = {
    'Sin tratamiento':      X_base.copy(),
    'Eliminar outliers':    X_base[~mask_out].copy(),
    'Winsorizar P1/P99':    X_base.assign(credit_amount=winsorized),
    'Log(credit_amount)':   X_base.assign(credit_amount=log_transformed),
    'Capping P5/P95':       X_base.assign(credit_amount=capped),
    'Agregar flag outlier': X_base.assign(is_outlier_ca=mask_out.astype(int)),
}

for name, X_strat in strategies.items():
    y_strat = y if name != 'Eliminar outliers' else y[~mask_out]
    lr = LogisticRegression(max_iter=1000, random_state=42)
    auc = cross_val_score(lr, X_strat, y_strat, cv=5, scoring='roc_auc').mean()
    delta = auc - cross_val_score(LogisticRegression(max_iter=1000, random_state=42),
                                   X_base, y, cv=5, scoring='roc_auc').mean()
    print(f"  {name:<25s}  AUC = {auc:.4f}  (Δ = {delta:+.4f})")


### 📝 Interpretación

En el German Credit Dataset, los outliers de `credit_amount` son **genuinos** (clientes de alto monto), no errores. Por eso:
- **Eliminar** outliers degrada ligeramente el AUC: se pierden registros informativos.
- **Log-transformar** y **winsorizar** producen AUC similares o levemente mejores: comprimen los extremos sin perder datos.
- **Agregar flag** como variable adicional puede capturar información predictiva del propio estatus de outlier.

**Regla de oro:** En datos financieros, *nunca* eliminar outliers sin confirmar que son errores. Los clientes atípicos suelen ser los más interesantes para el modelo.

---
## 9. Resumen


In [ ]:
print("╔══════════════════════════════════════════════════════════════════╗")
print("║              RESUMEN — OUTLIERS EN DATOS FINANCIEROS          ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  DETECCIÓN:                                                    ║")
print("║    · IQR: simple, no paramétrico. Contextualizar por segmento ║")
print("║    · Z-score MAD: robusto (breakdown 50%). |M| > 3.5          ║")
print("║    · Mahalanobis: multivariado, asume normalidad              ║")
print("║    · Isolation Forest: no paramétrico, multivariado           ║")
print("║  CLASIFICACIÓN:                                               ║")
print("║    · Error → corregir o eliminar                              ║")
print("║    · Genuino → winsorizar, log-transform o conservar          ║")
print("║    · Informativo → NUNCA eliminar (es el target)              ║")
print("║  TRATAMIENTO RECOMENDADO:                                     ║")
print("║    · Winsorización P1/P99 como primera opción                 ║")
print("║    · Log(1+x) para variables monetarias asimétricas           ║")
print("║    · Flag binario para capturar información del estatus       ║")
print("╚══════════════════════════════════════════════════════════════════╝")


---
> **Dr. Darío Ezequiel Díaz** · MMD UTN FRRo · 2026
